# **qlora_v1.ipynb**

PEFT-QLoRA example for fine-tuning with the FreedomIntelligence/medical-o1-reasoning-SFT dataset. This code is inspired by community projects and uses the lightweight DeepSeek-R1-Distill-Qwen-1.5B model, making it efficient even on a free T4 GPU.

**How It Works**

**Dataset Formatting**: The medical-o1-reasoning-SFT dataset contains Question, Complex_CoT, and Response fields. The code formats these into a chat template that encourages step-by-step reasoning (<think>...</think>) before the final answer .

**Efficient Training**: QLoRA dramatically reduces memory usage by quantizing the base model to 4-bit, allowing this 1.5B parameter model to be trained on a standard T4 GPU . The SFTTrainer from the trl library handles the supervised fine-tuning loop


**bitsandbytes** - The 4-bit **Quantization Engine
Purpose: Enables 4-bit quantization to dramatically reduce model memory usage.

**Why You Need It**:

Compresses a 1.5B parameter model from ~6GB (FP16) down to ~1.5GB (4-bit)

Allows training on free Colab GPUs (T4 with 16GB VRAM)

**Libraries Used:**

**trl - Transformer Reinforcement Learning**
Purpose: Provides high-level trainers and utilities for fine-tuning language models, especially with RLHF and instruction tuning.

Why You Need It:

Provides SFTTrainer - the most convenient way to fine-tune on instruction datasets

Handles tokenization, batching, and training loop automatically

Built specifically for conversational/chat models

In [1]:
# 1. Install required libraries
!pip install -q accelerate peft transformers datasets trl bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch

# 2. Configure model & QLoRA (4-bit quantization)
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# NF4 (Normal Float 4): Optimal 4-bit quantization for LLMs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [11]:
# 3. Load and format the datase
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", name="en", split="train[:1000]")  # Use subset for speed, 50, 500 and then remove it

In [12]:
def format_chat_template(example):
    return f"<|user|>\n{example['Question']}<|end|>\n<|assistant|>\n<think>\n{example['Complex_CoT']}\n</think>\n{example['Response']}"

In [13]:
# 4. Configure LoRA adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = prepare_model_for_kbit_training(model) # Reintroduced this line
# model = get_peft_model(model, lora_config) # This remains removed, as SFTTrainer handles it internally

In [14]:
# 5. Train with SFTTrainer
training_args = TrainingArguments(
    output_dir="./medical-qlora-results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=50,
    logging_steps=25,
    learning_rate=2e-4,
    warmup_steps=2, # Changed from warmup_ratio=0.03
    fp16=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config, # Added peft_config here
    formatting_func=format_chat_template,
)

trainer.train()
# 6. Save the fine-tuned adapter
model.save_pretrained("./medical-qlora-adapter")
print("✅ Fine-tuning complete. LoRA adapter saved.")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Step,Training Loss
25,2.364260
50,2.147965
75,2.052009
100,2.040304


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.364260
50,2.147965
75,2.052009
100,2.040304
125,1.984292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete. LoRA adapter saved.
